# MovieLens: item-based коллаборативная фильтрация

Рекомендательная модель на полном датасете MovieLens `ml-latest` (~33.8 млн оценок). Схожесть фильмов считается как корреляция Пирсона по общим пользователям; качество сравнивается с бейзлайном «средний рейтинг» по RMSE.

In [ ]:
# Зависимости: Java 11+ и PySpark
%pip install -q pyspark==3.5.1

## Данные и Spark-сессия

Разбиение на train/test 80/20.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from pyspark.sql.types import DoubleType
import pyspark.sql.functions as F
from pyspark.ml.evaluation import RegressionEvaluator
import builtins

# Оптимизированная конфигурация для полного датасета
spark = SparkSession.builder \
    .appName("ItemBasedCF_Full") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.memory.fraction", "0.8") \
    .getOrCreate()

# Загрузка полного датасета
ratings_path = "data/ml-latest/ratings.csv"
ratings = spark.read.csv(ratings_path, header=True, inferSchema=True)
# Разделение на train (80%) и test (20%)
train_init, test = ratings.randomSplit([0.8, 0.2], seed=42)

## Бейзлайн

Предсказываем всем фильмам средний рейтинг по train и считаем RMSE.

In [ ]:
# Средний рейтинг по обучающей выборке
avg_rating = train_init.agg(F.avg("rating")).collect()[0][0]
print(f"Средний рейтинг в train: {avg_rating:.4f}")

# Добавляем колонку prediction = avg_rating в test
test_with_baseline = test.withColumn("prediction", lit(avg_rating))

# Вычисляем RMSE для базовой модели
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
baseline_rmse = evaluator.evaluate(test_with_baseline)
print(f"Базовый RMSE: {baseline_rmse:.4f}")

# 2.1 Эффективное вычисление схожести между фильмами
movie_stats = train_init.groupBy("movieId").agg(
    F.avg("rating").alias("avg_rating"),
    F.count("rating").alias("rating_count")
).cache()

# Фильтруем только достаточно популярные фильмы для вычисления схожести
min_ratings = 100  # Минимум 100 оценок
popular_movies = movie_stats.filter(col("rating_count") >= min_ratings).select("movieId").cache()
popular_movies_count = popular_movies.count()
# Ограничиваем train только популярными фильмами для вычисления схожести
train_popular = train_init.join(popular_movies, "movieId").cache()

Средний рейтинг в train: 3.5425
Базовый RMSE: 1.0640


## Матрица схожести

Попарная корреляция Пирсона для популярных фильмов (≥100 оценок, ≥10 общих пользователей, |r| > 0.1), оценки пользователей собираются в словари для broadcast.

In [ ]:
def compute_item_similarities_batch(batch_size=50):
    from pyspark import SparkContext
    import pandas as pd
    import numpy as np

    sc = SparkContext.getOrCreate()

    popular_movies_list = [row['movieId'] for row in popular_movies.collect()]
    movie_ratings_data = {}

    for movie_id in popular_movies_list[:500]:  # Ограничиваем для демонстрации
        movie_ratings = train_popular.filter(col("movieId") == movie_id) \
                                    .select("userId", "rating") \
                                    .collect()
        movie_ratings_data[movie_id] = {row['userId']: row['rating'] for row in movie_ratings}

    # Вычисляем попарные схожести порциями
    similarities = []
    total_movies = len(popular_movies_list[:500])

    for i in range(0, total_movies, batch_size):
        end_idx = min(i + batch_size, total_movies)
        batch_movies = popular_movies_list[i:end_idx]

        for j, movie1 in enumerate(batch_movies):
            for k, movie2 in enumerate(batch_movies):
                if movie1 < movie2:  # Избегаем дубликатов
                    ratings1 = movie_ratings_data.get(movie1, {})
                    ratings2 = movie_ratings_data.get(movie2, {})

                    # Находим общих пользователей
                    common_users = set(ratings1.keys()) & set(ratings2.keys())

                    if len(common_users) >= 10:  # Минимум 10 общих пользователей
                        ratings1_vec = [ratings1[user] for user in common_users]
                        ratings2_vec = [ratings2[user] for user in common_users]

                        # Вычисляем корреляцию Пирсона
                        try:
                            corr = np.corrcoef(ratings1_vec, ratings2_vec)[0, 1]
                            if not np.isnan(corr) and abs(corr) > 0.1:
                                similarities.append((movie1, movie2, float(corr), len(common_users)))
                        except:
                            pass

    return similarities

# Вычисляем схожести
similarities = compute_item_similarities_batch(batch_size=30)
print(f"Вычислено {len(similarities)} значимых пар фильмов")

# Создаем словари для модели
movie_avg_dict = {}
for row in movie_stats.collect():
    movie_avg_dict[row['movieId']] = float(row['avg_rating'])

similarity_dict = {}
for movie1, movie2, corr, common_users in similarities:
    if movie1 not in similarity_dict:
        similarity_dict[movie1] = []
    if movie2 not in similarity_dict:
        similarity_dict[movie2] = []

    similarity_dict[movie1].append((movie2, corr, common_users))
    similarity_dict[movie2].append((movie1, corr, common_users))

# Сортируем по силе корреляции
for movie_id in similarity_dict:
    similarity_dict[movie_id].sort(key=lambda x: abs(x[1]), reverse=True)

print(f"Модель создана для {len(similarity_dict)} фильмов")

# Собираем оценки пользователей (выборочно для экономии памяти)
user_ratings_sample = train_init.sample(0.2, seed=42).select("userId", "movieId", "rating").collect()

user_ratings_dict = {}
for row in user_ratings_sample:
    user_id = row['userId']
    movie_id = row['movieId']
    rating = float(row['rating'])
    if user_id not in user_ratings_dict:
        user_ratings_dict[user_id] = {}
    user_ratings_dict[user_id][movie_id] = rating

print(f"Собраны оценки для {len(user_ratings_dict)} пользователей")

Вычислено 4018 значимых пар фильмов
Модель создана для 500 фильмов
Собраны оценки для 298307 пользователей


## Предсказание и оценка

UDF взвешивает оценки пользователя по 15 ближайшим фильмам. Вес равен |r| с поправкой на число общих пользователей, для отрицательной корреляции оценка отражается.

In [ ]:
from pyspark import SparkContext
sc = SparkContext.getOrCreate()

broadcast_movie_avg = sc.broadcast(movie_avg_dict)
broadcast_similarity = sc.broadcast(similarity_dict)
broadcast_user_ratings = sc.broadcast(user_ratings_dict)

def item_based_predict_full(user_id, movie_id):
    try:
        movie_avg = broadcast_movie_avg.value
        similarity_data = broadcast_similarity.value
        user_ratings = broadcast_user_ratings.value

        base_prediction = movie_avg.get(movie_id, avg_rating)

        if user_id not in user_ratings:
            return float(base_prediction)

        user_movie_ratings = user_ratings[user_id]

        if movie_id not in similarity_data:
            return float(base_prediction)

        similar_movies = similarity_data[movie_id][:15]
        weighted_sum = 0.0
        total_weight = 0.0
        count = 0

        for similar_movie, correlation, common_users in similar_movies:
            if similar_movie in user_movie_ratings:
                user_rating = user_movie_ratings[similar_movie]
                weight = abs(correlation) * min(1.0, common_users / 25.0)

                if correlation < 0:
                    adjusted_rating = 6.0 - user_rating
                else:
                    adjusted_rating = user_rating

                weighted_sum += weight * adjusted_rating
                total_weight += weight
                count += 1

        if count >= 2 and total_weight > 0:
            similar_contribution = weighted_sum / total_weight
            prediction = 0.8 * similar_contribution + 0.2 * base_prediction
        else:
            prediction = base_prediction

        return float(max(0.5, min(5.0, prediction)))

    except Exception as e:
        return float(movie_avg.get(movie_id, avg_rating))

# Применяем модель к тестовой выборке
predict_udf = F.udf(item_based_predict_full, DoubleType())

# Берем ограниченную выборку из теста для разумного времени выполнения
test_sample = test.sample(0.1, seed=42)
test_predictions = test_sample.withColumn("prediction", predict_udf(col("userId"), col("movieId")))

# Вычисляем итоговый RMSE
cf_rmse = evaluator.evaluate(test_predictions)

print(f"Средний рейтинг в train: {avg_rating:.4f}")
print(f"Базовый RMSE: {baseline_rmse:.4f}")
print(f"RMSE коллаборативной фильтрации по схожести обьектов: {cf_rmse:.4f}")

improvement = ((baseline_rmse - cf_rmse) / baseline_rmse) * 100
print(f"Улучшение RMSE: {improvement:.2f}%")
print(f"Оценок в train: {train_init.count():,}")
print(f"Оценок в test: {test.count():,}")

# Очистка
broadcast_movie_avg.unpersist()
broadcast_similarity.unpersist()
broadcast_user_ratings.unpersist()

movie_stats.unpersist()
popular_movies.unpersist()
train_popular.unpersist()

spark.stop()

Средний рейтинг в train: 3.5425
Базовый RMSE: 1.0640
RMSE коллаборативной фильтрации по схожести обьектов: 0.9667
Улучшение RMSE: 9.14%
Оценок в train: 27,066,368
Оценок в test: 6,765,794


## Итог

| Модель | RMSE |
|---|---|
| Средний рейтинг (бейзлайн) | 1.0640 |
| Item-based CF | **0.9667** |

Улучшение RMSE составило **9.14%** относительно бейзлайна.